In [2]:
import polars as pl


pdb = pl.read_parquet(
    "/tgen_labs/altin/alphafold3/workspace/tcrtrifold-experiments/data/pdb/triad/staged/pdb_triad.conf.parquet"
)

old_pdb = pl.read_csv(
    "/tgen_labs/altin/alphafold3/workspace/tcrtrifold-experiments/data/pdb/raw/old_pdb.csv"
)

supptable = pl.read_csv(
    "/tgen_labs/altin/alphafold3/workspace/tcrtrifold-experiments/data/cresta/raw/SuppTable1_raw.csv"
)

In [ ]:
supptable.filter(pl.col("TCRtype") == "PDB").select("peptide").unique().select(
    pl.col("peptide").str.len_chars().mean()
)

peptide
f64
14.0


all of john's PDB IDs were in the training data

In [10]:
pdb.join(old_pdb, on="pdb").select("replication").unique()

replication
bool
true


In [14]:
import requests
from mdaf3.FeatureExtraction import split_apply_combine
from datetime import datetime, timezone


def extract_pdb_date(row):
    r = requests.get("https://data.rcsb.org/rest/v1/core/entry/" + row["pdb"])
    r.raise_for_status()
    new_row = row.copy()
    new_row["pdb_date"] = r.json()["rcsb_accession_info"]["initial_release_date"]

    return new_row


def get_pdb_date(df):
    df = split_apply_combine(
        df,
        extract_pdb_date,
        chunksize=50,
    ).with_columns(pl.col("pdb_date").str.to_datetime().alias("pdb_date"))
    return df


old_pdb = get_pdb_date(old_pdb)

cutoff = pl.lit(datetime(2023, 1, 12, tzinfo=timezone.utc))

old_pdb = old_pdb.with_columns(
    pl.when(pl.col("pdb_date") > cutoff)
    .then(pl.lit(False))
    .otherwise(pl.lit(True))
    .alias("in_training")
)

100%|██████████| 29/29 [00:01<00:00, 19.40it/s]


In [17]:
old_pdb.filter(~pl.col("in_training"))

pdb,,pdb_date,in_training
str,null,"datetime[μs, UTC]",bool
